<a href="https://colab.research.google.com/github/rishabhucb/court-keypoints/blob/main/day2_dataset_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install roboflow ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 361.2/361.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 133.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 12.5 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2


In [3]:
cd


/root


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls /content/drive/MyDrive

'Colab Notebooks'			   Rishabh_Singh_Profile.docx
'Copy of AIM Projects Document F26.gdoc'  'YOLO model Baskball court .gdoc'
 court_keypoints


In [6]:
!ls /content/drive/MyDrive/court_keypoints

basketball-court-keypoints-1  coco_export


In [7]:
DATA_DIR = '/content/drive/MyDrive/court_keypoints/basketball-court-keypoints-1'
%cd {DATA_DIR}
!ls

/content/drive/MyDrive/court_keypoints/basketball-court-keypoints-1
data.yaml  README.roboflow.txt	test  train  valid


In [8]:
import yaml, glob, os, collections
cfg = yaml.safe_load(open(f'{DATA_DIR}/data.yaml'))
print("Classes:", cfg['names'])
print("kpt_shape:", cfg['kpt_shape'])
print("flip_idx:", cfg.get('flip_idx', 'NOT PRESENT'))

n_kpts, kdim = cfg['kpt_shape']
for split in ['train', 'valid', 'test']:
    labels = glob.glob(f'{DATA_DIR}/{split}/labels/*.txt')
    visible = collections.Counter()
    for f in labels:
        for line in open(f):
            v = list(map(float, line.split()))[5:]
            for i in range(n_kpts):
                if kdim == 2 or v[i*kdim + 2] > 0:
                    visible[i] += 1
    print(f"\n{split}: {len(labels)} images")
    for i in range(n_kpts):
        print(f"  kpt {i:2d}: visible in {visible[i]/max(len(labels),1):.0%} of images")

Classes: ['basketball_court']
kpt_shape: [13, 3]
flip_idx: [0, 1, 2, 6, 5, 4, 3, 8, 7, 9, 11, 10, 12]

train: 150 images
  kpt  0: visible in 60% of images
  kpt  1: visible in 37% of images
  kpt  2: visible in 58% of images
  kpt  3: visible in 100% of images
  kpt  4: visible in 100% of images
  kpt  5: visible in 100% of images
  kpt  6: visible in 100% of images
  kpt  7: visible in 100% of images
  kpt  8: visible in 100% of images
  kpt  9: visible in 100% of images
  kpt 10: visible in 100% of images
  kpt 11: visible in 37% of images
  kpt 12: visible in 59% of images

valid: 22 images
  kpt  0: visible in 73% of images
  kpt  1: visible in 27% of images
  kpt  2: visible in 64% of images
  kpt  3: visible in 91% of images
  kpt  4: visible in 95% of images
  kpt  5: visible in 100% of images
  kpt  6: visible in 100% of images
  kpt  7: visible in 100% of images
  kpt  8: visible in 100% of images
  kpt  9: visible in 100% of images
  kpt 10: visible in 95% of images
  kpt 11

In [9]:
from google.colab import userdata
from roboflow import Roboflow
import json, glob

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("rishabh-ucb-gmail-com").project("basketball-court-keypoints-l4d5o-puuiw")
coco = project.version(1).download("coco", location="/content/drive/MyDrive/court_keypoints/coco_export")

ann = json.load(open(glob.glob(f"{coco.location}/train/_annotations.coco.json")[0]))
for c in ann['categories']:
    if c.get('keypoints'):
        print("Class:", c['name'])
        for i, name in enumerate(c['keypoints']):
            print(f"  ID {i:2d} -> {name}")

loading Roboflow workspace...
loading Roboflow project...
Class: basketball_court
  ID  0 -> baseline_top_corner
  ID  1 -> baseline_bottom_corner
  ID  2 -> sideline_top_halfcourt
  ID  3 -> paint_top_left
  ID  4 -> paint_bottom_left
  ID  5 -> paint_bottom_right
  ID  6 -> paint_top_right
  ID  7 -> threept_arc_top_baseline
  ID  8 -> threept_arc_top_wing
  ID  9 -> threept_arc_apex
  ID 10 -> threept_arc_bottom_wing
  ID 11 -> threept_arc_bottom_baseline
  ID 12 -> sideline_bottom_halfcourt


In [10]:
!ls {DATA_DIR}/train/images | head

frame_001_jpg.rf.61b25616af2c40585cdd2be27663610e.jpg
frame_002_jpg.rf.34e10a7f6f01ba06186f7d8f6042ea45.jpg
frame_005_jpg.rf.7f5c9841389b51c3f779acd0caaba1ee.jpg
frame_006_jpg.rf.383c97b226eb9be21c349b5615fd5390.jpg
frame_007_jpg.rf.f139c73cd6ae013f4b4e6cd146d0e79b.jpg
frame_008_jpg.rf.933d8a4e72de71ae2a0e540e22abe059.jpg
frame_009_jpg.rf.3094e523bf9dd747568deeae3a780794.jpg
frame_013_jpg.rf.4547dbcb1279b6b10107c3a6cf0e6fc1.jpg
frame_014_jpg.rf.e9dddc87139c868a0bc8e8a79ec1300e.jpg
frame_015_jpg.rf.28b797b8d012759f769c193ce4a7e0ef.jpg


In [11]:
import glob, os, collections

def source(fname):
    base = os.path.basename(fname).split('.rf.')[0]   # original file name
    return base.rsplit('_', 1)[0]                     # strip frame-number suffix

where = collections.defaultdict(set)
for split in ['train', 'valid', 'test']:
    for f in glob.glob(f'{DATA_DIR}/{split}/images/*'):
        where[source(f)].add(split)

print("Unique sources:", len(where))
leaks = {s: sp for s, sp in where.items() if len(sp) > 1}
print("Sources in more than one split:", len(leaks))
for s, sp in list(leaks.items())[:10]:
    print("  ", s, sorted(sp))

Unique sources: 184
Sources in more than one split: 0
